# Montmory on TPU: conformance, then measurements

This notebook contains the complete kit. In Colab, select a TPU runtime, then run the cells in order. It does not reserve or rent resources. TPU access depends on the account and quotas.

The prototype has been checked on x86 CPU and in interpreted Pallas mode. TPU-targeted export succeeded for the portable path. **Native TPU execution and speed remain to be measured.**

The protocol preserves the Montmory stream at T=2048,D=4096. It compares 64-bit integer outputs, then JAX implementation throughput. CPU results do not predict TPU results.

English reading edition of the [original French notebook](Montmory_TPU_validation.ipynb). The paragraph above preserves the historical status when that notebook was prepared; subsequent native results are documented in the [current evidence register](../docs/en/evidence.md). Executable cells, embedded kit and diagnostic messages are unchanged. [Notebook guide](README.md).

In [ ]:
import base64, hashlib, io, os, tarfile
from pathlib import Path

ROOT = Path(os.environ.get("MONTMORY_NOTEBOOK_ROOT", "/content")).resolve()
ROOT.mkdir(parents=True, exist_ok=True)
PAYLOAD = """..."""
archive_bytes = base64.b64decode(PAYLOAD)
assert hashlib.sha256(archive_bytes).hexdigest() == "92baf16a55f3822298d3b42b21677a0b5828343640e7281a7d826a0db624f172"
with tarfile.open(fileobj=io.BytesIO(archive_bytes), mode="r:gz") as archive:
    for member in archive.getmembers():
        target = (ROOT / member.name).resolve()
        if ROOT not in target.parents or not member.isfile():
            raise ValueError("Unexpected archive member")
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.extractfile(member).read())
KIT = ROOT / "montmory-jax-tpu"
print("Kit vérifié et extrait :", KIT)


## Install the specified versions


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, str(KIT / "check_manifest.py")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(KIT / "requirements-tpu.txt")], check=True)


## Run on TPU
Compilation precedes measurement. If an error occurs, proceed to the download cell.


In [ ]:
import datetime

# False = protocole complet : 5 seeds, 7 répétitions, deux volumes.
QUICK = False
# RBG est contrôlé séparément, sur le même backend uniquement.
INCLUDE_RBG = True

stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RESULTS = KIT / "results" / (stamp + "-colab-tpu")
RESULT_ARCHIVE = RESULTS.parent / (RESULTS.name + ".tar.gz")
command = [sys.executable, "-u", str(KIT / "run_tpu.py"), "--out", str(RESULTS)]
if QUICK:
    command.append("--quick")
if INCLUDE_RBG:
    command.append("--include-rbg")
result = subprocess.run(command, cwd=KIT)
print("Archive de résultats / diagnostics :", RESULT_ARCHIVE)
if result.returncode:
    raise RuntimeError("Essai non validé. Télécharger les diagnostics avec la cellule suivante.")


## Download results, even after an error


In [ ]:
from google.colab import files
if not RESULT_ARCHIVE.is_file():
    raise FileNotFoundError("Exécuter la cellule de tests avant de télécharger les diagnostics.")
files.download(str(RESULT_ARCHIVE))


## Read the result

Success is established only if `run-status.json` reports `pass`. If an error occurs, the archive contains the log and completed reports. TPU export or a CPU test is not native TPU execution. The script requires the TPU backend.

Each measurement waits for computation to finish. Compilation, key preparation and output verification are outside timing. All timed results are checked. Pallas configurations and all measurements remain in the CSV files.

Threefry/Philox streams are those of the pinned JAX API, checked against CPU hashes. The RBG option checks repeatability only on the same backend; it does not guarantee identical bits across TPU generations. No new TestU01/TACM is run, and Montmory `split`/`fold_in` are not implemented.
